# t0207: Elsys tpc5 + NI npz → Labquake Explorer HDF5

Preprocessing for an experiment recorded on two systems.

* **Elsys TranAX** (`.tpc5`, ECR dual mode): block 1 is a 2 kHz continuous record of
  the whole run, blocks 2…N are 2 MHz records around each PZT trigger (0.1 s before,
  0.5 s after). All blocks share the Elsys clock,
  `t = trigger_time + (i - trigger_sample) / sample_rate`.
* **National Instruments** (`.npz`, run5 only): the mechanical channels recorded
  continuously at 500 kHz on the NI clock, with `trigger_sample_index` marking the
  sample at which the first Elsys trigger arrived.

Runs 1–4 had every sensor on the Elsys and use the tpc5-only recipe of
`t0211_tpc5.ipynb`. In run5 the mechanical sensors were moved to the NI logger
(the Elsys channels A5–A7 and B1–B8 read 0 V), so that run is built from the NI
record, decimated to the Elsys continuous rate, with the tpc5 kept as the source of
the PZT trigger blocks. The run's time axis is the NI clock; the Elsys clock is
shifted onto it by `strain.time_offset`.

Run sheet (2026-08-20, 21 °C, 57 % RH, flow rate 8 ml/min, operators 朱展穎、張德權):

| Elsys channel | sensor        | range             |
|---------------|---------------|-------------------|
| A1–A4         | PZT           | ±0.25 V (run1–4)  |
| A5, A6        | pressure      | 0–10 V            |
| A7            | LVDT          | −6 to 14 V        |
| B1–B8         | eddy current  | ±25 V             |

Remark on the sheet: after run2 block 20 the E02 slip channel is suspect.

In [ ]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
%matplotlib widget

# repository root, in case labquake_explorer is not pip-installed
sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from labquake_explorer.utils import tpc5
from labquake_explorer.utils.ni_npz import open_ni_npz
from labquake_explorer.data.data_manager import DataManager

## Paths

NI files are paired with tpc5 files by the `runN` token in their names.

In [ ]:
EXPERIMENT = 't0207'
EXPERIMENT_DIR = Path('/Users/hueyke/Library/CloudStorage/SynologyDrive-data/Experiment/T0207')
DATA_DIR = EXPERIMENT_DIR / f'{EXPERIMENT}.exp' / 'data'

# Output read by Labquake Explorer. Raw-file paths are stored relative to this file,
# so keep it next to the raw data (or move them together).
OUTPUT_FILE = EXPERIMENT_DIR / f'{EXPERIMENT}.h5'

RUN_NAME = re.compile(rf'{EXPERIMENT}_(?P<index>\d+)_(?P<sigma>[\d.]+)MPa_(?P<run>run\d+)')
TPC5_FILES = sorted(DATA_DIR.glob(f'{EXPERIMENT}_*.tpc5'))
NI_FILES = {m.group(1): p for p in DATA_DIR.glob('*.npz') for m in [re.search(r'(run\d+)', p.name)] if m}

rows = []
for p in TPC5_FILES:
    run = RUN_NAME.match(p.stem)['run']
    ni = NI_FILES.get(run)
    rows.append({'run': run, 'tpc5': p.name, 'tpc5 (MB)': round(p.stat().st_size / 1e6),
                 'ni npz': ni.name if ni else '', 'ni (MB)': round(ni.stat().st_size / 1e6) if ni else ''})
pd.DataFrame(rows)

## What is in the files

NI record metadata and the tpc5 block table of run5 (metadata only).

In [ ]:
ni = open_ni_npz(NI_FILES['run5'])
info = ni.as_dict()
display(pd.Series({k: v for k, v in info.items() if k != 'member_offsets'}))
with h5py.File(DATA_DIR / 't0207_04_16MPa_run5.tpc5', 'r') as f:
    display(pd.DataFrame(tpc5.list_channels(f)).head(8))
    display(pd.DataFrame([b.as_dict() for b in tpc5.list_blocks(f)]))

## Channel roles, calibration and decimation

`ELSYS_CHANNELS` and `NI_CHANNELS` map each recorder's channel to the field name
stored in the run (volts). The NI roles come from the group research log entry for
this run ("[T207] NI-6451 DAQ Trial Run", 2026-08-20): AI-0 normal-stress voltage,
AI-1 shear-stress voltage, AI-2 LVDT, AI-3 an eddy-current sensor with a 9 V dry cell
wired in series as an offset trial (so it reads about 9 V above the others), AI-4 to
AI-10 the usual eddy-current sensors. The NI USB-6451 ran at 500 kS/s per channel on
a ±10 V input range and captured the Elsys TTL trigger on port0/line0; the Elsys only
emits that pulse on its first trigger, which is why `trigger_sample_index` marks the
first Elsys trigger block. The ±10 V input is narrower than the eddy-current sensors'
−25 to 0 V output, so large slips can clip (run5 reaches −9.8 V on ai7 without clipping).

Stress conversions use the lab's pressure-transducer calibration for a 1D fault with
a 5 cm cross-section (Tseng 2026, NTU thesis, Appendix B.3.3 and Table B.1):
`sigma_n = 4.507 V - 0.410` MPa on A5 (runs 1–4 reproduce their nominal 10–16 MPa
within 2 %) and `V = 0.109 tau` on A6.

The eddy-current sensors follow the lab's stage calibration, `d (mm) = slope * V +
intercept`, with slopes of -94.7, -96.6 and -95.4 um/V for the sensors on B1, B2 and
B3 (R² >= 0.9997, RMSE <= 10 um, travel up to 2 mm); the other five sensors get the
mean slope until they are calibrated. The intercepts describe the calibration jig, so
each `slip_k` channel is zeroed on the first 0.5 s of the run instead, which is also
how the experiments are run (sensors zeroed after the run-in). The slope is negative:
the voltage falls as the gap opens, so slip accumulates positive. `displacement`, the
field the Event Analyzer uses, is the mean of the eight slip channels by default
(`DISPLACEMENT_SOURCE`). The LVDT slope is still unknown and stays in volts.

`DECIMATION` is the block-mean factor applied to the 500 kHz NI record for the run's
time history (250 → 2 kHz, the Elsys continuous rate). The full-rate record stays
available through the `ni` reference stored in the run.

In [ ]:
ELSYS_CHANNELS = {
    'A1': 'pzt_1', 'A2': 'pzt_2', 'A3': 'pzt_3', 'A4': 'pzt_4',
    'A5': 'pressure_1', 'A6': 'pressure_2',
    'A7': 'lvdt',
    'B1': 'eddy_1', 'B2': 'eddy_2', 'B3': 'eddy_3', 'B4': 'eddy_4',
    'B5': 'eddy_5', 'B6': 'eddy_6', 'B7': 'eddy_7', 'B8': 'eddy_8',
}
NI_CHANNELS = {   # research log, [T207] NI-6451 DAQ Trial Run (2026-08-20)
    'ai0': 'pressure_1', 'ai1': 'pressure_2', 'ai2': 'lvdt',
    'ai3': 'eddy_1',      # 9 V dry cell in series as an offset trial: reads ~9 V above the others
    'ai4': 'eddy_2', 'ai5': 'eddy_3', 'ai6': 'eddy_4',
    'ai7': 'eddy_5', 'ai8': 'eddy_6', 'ai9': 'eddy_7', 'ai10': 'eddy_8',
}
NI_CHANNEL_NOTES = {'ai3': '9 V dry cell wired in series before the DAQ (offset trial)'}
PZT_FIELDS = ('pzt_1', 'pzt_2', 'pzt_3', 'pzt_4')

# 1D fault, 5 cm cross-section (Tseng 2026, Appendix B.3.3 / Table B.1)
NORMAL_V_PER_MPA, NORMAL_OFFSET_V = 0.222, 0.091   # V = 0.222 sigma_n + 0.091
SHEAR_V_PER_MPA = 0.109                            # V = 0.109 tau (no intercept given)

CALIBRATION = {
    # field: source volts, factor, offset, unit of the result  (value = factor * V + offset)
    'normal_stress':   dict(source='pressure_1', factor=1 / NORMAL_V_PER_MPA,
                            offset=-NORMAL_OFFSET_V / NORMAL_V_PER_MPA, unit='MPa'),
    'shear_stress':    dict(source='pressure_2', factor=1 / SHEAR_V_PER_MPA, offset=0.0, unit='MPa'),
    'LP_displacement': dict(source='lvdt',       factor=1.0, offset=0.0, unit='V'),  # um per V: to fill in
}

# Eddy-current sensors: the lab calibration gives d = slope * V + intercept with d in
# mm (three sensors, R^2 >= 0.9997, RMSE <= 10 um, travel up to 2 mm). The intercepts
# belong to the calibration jig, so every slip channel is zeroed on the first
# ZERO_WINDOW_S of the run instead; sensors without their own line use the mean slope.
EDDY_MM_PER_V = {'eddy_1': -0.094727676, 'eddy_2': -0.096627292, 'eddy_3': -0.095384964}   # B1, B2, B3
EDDY_MM_PER_V_DEFAULT = float(np.mean(list(EDDY_MM_PER_V.values())))
ZERO_WINDOW_S = 0.5             # slip channels are zeroed on the mean of this first interval
DISPLACEMENT_SOURCE = 'mean'    # 'mean' of all slip channels, or one of them, e.g. 'slip_5'


DECIMATION = 250            # 500 kHz -> 2 kHz
ALIGNMENT_CHANNEL = 'ai7'   # eddy-current channel with the clearest slip steps

## Clock alignment

`trigger_sample_index` in the NI file is the sample at which the first Elsys trigger
arrived, so `offset = ni.trigger_time - first Elsys trigger_time` puts the Elsys
clock on the NI clock (`t_run = t_elsys + offset`). The check below looks, for every
Elsys trigger block, for the largest 20 ms change of an eddy-current channel in the
NI record within ±1 s of the shifted trigger time; slip should follow the PZT
trigger by tens of milliseconds, never precede it.

In [ ]:
def clock_offset(ni, blocks):
    trig = tpc5.trigger_blocks(blocks)
    if ni.trigger_time is None or not trig:
        raise ValueError('need an NI trigger_sample_index and at least one Elsys trigger block')
    return ni.trigger_time - trig[0].trigger_time


def largest_step(t, y, t_center, half=1.0, width=0.02):
    m = (t >= t_center - half) & (t <= t_center + half)
    tt, yy = t[m], y[m]
    if tt.size < 10:
        return np.nan, np.nan
    w = max(1, int(round(width / (tt[1] - tt[0]))))
    d = yy[w:] - yy[:-w]
    i = int(np.argmax(np.abs(d)))
    return float(tt[i] + 0.5 * w * (tt[1] - tt[0])), float(d[i])


def check_alignment(ni, blocks, offset, channel=ALIGNMENT_CHANNEL, step_min_v=0.05):
    rows = []
    for b in tpc5.trigger_blocks(blocks):
        t_shift = b.trigger_time + offset
        if not (ni.contains(t_shift - 1.0) and ni.contains(t_shift + 1.0)):
            rows.append({'block': b.block, 't_elsys': b.trigger_time, 't_run': t_shift, 'slip step': 'outside NI record'})
            continue
        t, d = ni.read_window(t_shift - 1.0, t_shift + 1.0, [channel])
        n = d.shape[1] // DECIMATION * DECIMATION
        td = t[:n].reshape(-1, DECIMATION).mean(axis=1)
        yd = d[0, :n].reshape(-1, DECIMATION).mean(axis=1)
        t_step, dv = largest_step(td, yd, t_shift)
        rows.append({'block': b.block, 't_elsys': b.trigger_time, 't_run': t_shift,
                     'slip step': 'yes' if abs(dv) >= step_min_v else 'none > %.2f V' % step_min_v,
                     'step t_run': t_step if abs(dv) >= step_min_v else np.nan,
                     'lag (ms)': 1e3 * (t_step - t_shift) if abs(dv) >= step_min_v else np.nan,
                     'dV': dv})
    return pd.DataFrame(rows)


with h5py.File(DATA_DIR / 't0207_04_16MPa_run5.tpc5', 'r') as f:
    run5_blocks = tpc5.list_blocks(f)
CLOCK_OFFSET_S = clock_offset(ni, run5_blocks)     # override here if the check disagrees
print(f'Elsys -> NI clock offset: {CLOCK_OFFSET_S:.4f} s')
alignment = check_alignment(ni, run5_blocks, CLOCK_OFFSET_S)
display(alignment)
lags = alignment['lag (ms)'].dropna()
print(f'slip steps found for {lags.size} of {len(alignment)} triggers; lag after the PZT trigger: '
      f'{lags.min():.0f}..{lags.max():.0f} ms' if lags.size else 'no slip steps found')

## Read the runs

In [ ]:
def run_metadata(path: Path) -> dict:
    m = RUN_NAME.match(path.stem)
    return {'name': m['run'] if m else path.stem,
            'normal_stress_level': float(m['sigma']) if m else float('nan')}


def add_slip_channels(run: dict, units: dict) -> None:
    """slip_k (um, slip positive, zeroed at the run start) from each eddy_k voltage,
    and displacement from them according to DISPLACEMENT_SOURCE."""
    dt = float(np.median(np.diff(run['time'][:2000])))
    n0 = max(1, int(round(ZERO_WINDOW_S / dt)))
    slips = []
    for k in range(1, 9):
        src = f'eddy_{k}'
        if src not in run:
            continue
        v = run[src].astype(np.float64)
        slope_um_per_v = 1000.0 * EDDY_MM_PER_V.get(src, EDDY_MM_PER_V_DEFAULT)
        s = slope_um_per_v * (v - np.nanmean(v[:n0]))
        run[f'slip_{k}'] = s.astype(np.float32)
        units[f'slip_{k}'] = 'um'
        slips.append(s)
    if slips:
        d = np.mean(slips, axis=0) if DISPLACEMENT_SOURCE == 'mean' else run[DISPLACEMENT_SOURCE]
        run['displacement'] = np.asarray(d, dtype=np.float32)
        units['displacement'] = 'um'
    run['calibration'] = {
        'normal_stress': f'sigma_n (MPa) = V / {NORMAL_V_PER_MPA} - {NORMAL_OFFSET_V / NORMAL_V_PER_MPA:.3f}',
        'shear_stress': f'tau (MPa) = V / {SHEAR_V_PER_MPA}',
        'eddy_mm_per_v': {f'eddy_{k}': EDDY_MM_PER_V.get(f'eddy_{k}', EDDY_MM_PER_V_DEFAULT) for k in range(1, 9)},
        'slip_zero_window_s': ZERO_WINDOW_S,
        'displacement_source': DISPLACEMENT_SOURCE,
    }


def apply_calibration(run: dict, units: dict) -> None:
    for field, cal in CALIBRATION.items():
        if cal['source'] not in run:
            continue
        run[field] = (cal['factor'] * run[cal['source']].astype(np.float64) + cal['offset']).astype(np.float32)
        units[field] = cal['unit']
    if units.get('shear_stress') == units.get('normal_stress') != 'V':
        with np.errstate(divide='ignore', invalid='ignore'):
            run['friction'] = run['shear_stress'] / run['normal_stress']
        units['friction'] = ''
    add_slip_channels(run, units)
    run['units'] = units


def strain_reference(path: Path, rec: dict, time_offset: float) -> dict:
    fields = [ELSYS_CHANNELS.get(c['name'].split()[0], c['name']) for c in rec['channels']]
    ref = {
        'format': 'tpc5',
        'filename': os.path.relpath(path, OUTPUT_FILE.parent),
        'time_offset': float(time_offset),           # t_run = t_elsys + time_offset
        'channel_numbers': [c['number'] for c in rec['channels']],
        'channel_names': [c['name'] for c in rec['channels']],
        'fields': fields,
        'continuous': rec['block'].as_dict(),
        'blocks': tpc5.block_table(rec['trigger_blocks']),
    }
    ref['blocks']['trigger_time_run'] = [b.trigger_time + time_offset for b in rec['trigger_blocks']]
    return ref


def load_run_tpc5(path: Path) -> dict:
    # every sensor on the Elsys: block 1 is the run's time history
    rec = tpc5.read_continuous(path)
    run = run_metadata(path)
    run.update({'file': path.name, 'start_time': rec['start_time'], 'time': rec['time'].astype(np.float64)})
    units = {}
    for info, volts in zip(rec['channels'], rec['data']):
        field = ELSYS_CHANNELS.get(info['name'].split()[0], info['name'])
        run[field] = volts
        units[field] = 'V'
    apply_calibration(run, units)
    run['strain'] = strain_reference(path, rec, 0.0)
    return run


def load_run_tpc5_ni(tpc5_path: Path, ni_path: Path) -> dict:
    # mechanical channels on the NI logger, PZT on the Elsys
    ni = open_ni_npz(ni_path)
    rec = tpc5.read_continuous(tpc5_path)
    offset = clock_offset(ni, [rec['block']] + rec['trigger_blocks'])
    t_run, decimated = ni.decimate(DECIMATION)
    run = run_metadata(tpc5_path)
    run.update({'file': tpc5_path.name, 'ni_file': ni_path.name, 'start_time': rec['start_time'],
                'time': t_run.astype(np.float64)})
    units = {}
    for name, volts in zip(ni.channels, decimated):
        field = NI_CHANNELS.get(name, name)
        run[field] = volts
        units[field] = 'V'
    # Elsys continuous PZT channels, moved onto the run clock
    t_elsys = rec['time'] + offset
    for info, volts in zip(rec['channels'], rec['data']):
        field = ELSYS_CHANNELS.get(info['name'].split()[0], info['name'])
        if field in PZT_FIELDS:
            run[field] = np.interp(t_run, t_elsys, volts.astype(np.float64), left=np.nan, right=np.nan).astype(np.float32)
            units[field] = 'V'
    apply_calibration(run, units)
    run['ni'] = dict(ni.as_dict(), filename=os.path.relpath(ni_path, OUTPUT_FILE.parent),
                     fields=[NI_CHANNELS.get(c, c) for c in ni.channels], decimation=DECIMATION, time_offset=0.0,
                     channel_notes=dict(NI_CHANNEL_NOTES), input_range_v=10.0)
    run['strain'] = strain_reference(tpc5_path, rec, offset)
    run['strain']['clock_offset_method'] = 'NI trigger_sample_index = first Elsys trigger'
    return run


runs = []
for p in TPC5_FILES:
    run = RUN_NAME.match(p.stem)['run']
    runs.append(load_run_tpc5_ni(p, NI_FILES[run]) if run in NI_FILES else load_run_tpc5(p))

pd.DataFrame([{
    'name': r['name'], 'source': 'tpc5 + NI' if 'ni' in r else 'tpc5',
    'sigma_n nominal (MPa)': r['normal_stress_level'],
    'sigma_n measured (MPa)': float(np.nanmean(r['normal_stress'])),
    'samples': r['time'].size, 'duration (s)': float(r['time'][-1] - r['time'][0]),
    'trigger blocks': len(r['strain']['blocks']['block']),
    'Elsys -> run offset (s)': r['strain']['time_offset'],
    'displacement end (um)': float(r['displacement'][-1]),
} for r in runs])

## Overview

Stresses and LVDT of every run; shaded spans are the Elsys trigger blocks on the run's clock.

In [ ]:
fig, axs = plt.subplots(len(runs), 1, figsize=(10, 2.2 * len(runs)), sharex=True)
for ax, run in zip(np.atleast_1d(axs), runs):
    ax.plot(run['time'], run['normal_stress'], lw=0.6, label=f"normal_stress ({run['units']['normal_stress']})")
    ax.plot(run['time'], run['shear_stress'], lw=0.6, label=f"shear_stress ({run['units']['shear_stress']})")
    ax2 = ax.twinx()
    ax2.plot(run['time'], run['LP_displacement'], lw=0.6, color='C2')
    ax2.set_ylabel(f"LVDT ({run['units']['LP_displacement']})", color='C2')
    blocks = run['strain']['blocks']
    for a, b in zip(blocks['start'], blocks['end']):
        ax.axvspan(a + run['strain']['time_offset'], b + run['strain']['time_offset'], color='0.8', zorder=-10)
    ax.set_ylabel(f"{run['name']}  {run['normal_stress_level']:g} MPa" + ("  (NI)" if 'ni' in run else ""))
    ax.legend(loc='upper left', fontsize='small')
axs[-1].set_xlabel('time (s)')
fig.tight_layout()

## Run5 check: NI slip against Elsys triggers

The eddy-current channel used for the alignment, decimated to the run rate, with the
Elsys trigger times after the clock shift. Every PZT trigger that produced a slip
step should sit just before that step.

In [ ]:
run5 = next(r for r in runs if 'ni' in r)
field = NI_CHANNELS[ALIGNMENT_CHANNEL]
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(run5['time'], run5[field], lw=0.6, label=f'{field} ({ALIGNMENT_CHANNEL}, NI, {1 / DECIMATION * ni.sample_rate:g} Hz)')
for t in run5['strain']['blocks']['trigger_time_run']:
    ax.axvline(t, color='k', ls=':', lw=0.8)
ax.set_xlabel('time (s, NI clock)'); ax.set_ylabel('V'); ax.legend(loc='upper right')
ax.set_title('run5: eddy-current slip vs Elsys PZT triggers (dotted) after the clock shift')

## Experiment record and save

In [ ]:
experiment = {
    'name': EXPERIMENT,
    'date': '2026-08-20',
    'operators': ['朱展穎', '張德權'],
    'temperature_C': 21.0,
    'humidity_percent': 57.0,
    'flow_rate_ml_min': 8.0,
    'acquisition': ('Runs 1-4: Elsys TranAX ECR dual mode, block 1 continuous at 2 kHz, trigger blocks at 2 MHz '
                    '(0.1 s pre, 0.5 s post); A1-A4 PZT, A5-A6 pressure, A7 LVDT, B1-B8 eddy current. '
                    'Run 5: mechanical channels on a National Instruments logger at 500 kHz (ai0-ai10, '
                    'decimated by block means for the time history), PZT on the Elsys; run time axis is the NI clock.'),
    'notes': 'Run sheet: after run2 block 20 the E02 slip channel is suspect.',
    'runs': runs,
}

dm = DataManager()
dm.data = experiment
dm.save_file(OUTPUT_FILE)
print(f'{OUTPUT_FILE}  {OUTPUT_FILE.stat().st_size / 1e6:.1f} MB')

## Check the file the way the explorer loads it

In [ ]:
check = DataManager()
check.load_file(OUTPUT_FILE)
rows = []
for run in check.get_data('runs'):
    tpc5_path = OUTPUT_FILE.parent / run['strain']['filename']
    ni_path = OUTPUT_FILE.parent / run['ni']['filename'] if 'ni' in run else None
    rows.append({
        'run': run['name'], 'samples': len(run['time']),
        'fields': len([k for k, v in run.items() if isinstance(v, np.ndarray)]),
        'trigger blocks': len(run['strain']['blocks']['block']),
        'tpc5 found': tpc5_path.exists(),
        'ni found': ni_path.exists() if ni_path else '-',
        'offset (s)': run['strain']['time_offset'],
    })
pd.DataFrame(rows)

## Next

1. Open `t0207.h5` in Labquake Explorer, right-click `runs/[i]/shear_stress` → **Pick Events**, then `event_indices` → **Extract Events**.
2. Event extraction for `strain.format == 'tpc5'` (reading the trigger block that holds an event time, shifted by `time_offset`) and for the `ni` reference (reading the 500 kHz mechanical window straight from the npz) is the next step on the explorer side.